# SmartFood: ejecucion secuencial de notebooks 01-05

Este notebook ejecuta preparación, análisis exploratorio, baseline, NMF y SVM en ese orden. Cada etapa utiliza un kernel nuevo del mismo Python que seleccionaste aqui. Si una etapa falla, se detiene la secuencia.

**Antes de comenzar:** ejecuta 00_configuracion_entorno.ipynb en este mismo Python, guarda los cambios y cierra los notebooks 01-05 para evitar sobrescribir cambios sin guardar. Deben estar disponibles los Parquet de data/raw/.

Se actualizan las salidas de los notebooks que terminan correctamente y los artefactos de datos, modelos y reportes que generan sus celdas. No se vacía reports/ automáticamente. El limite configurado se aplica a cada celda, no a toda la secuencia.

Una ejecucion sin errores no garantiza que los diagnosticos impresos hayan pasado ni que los modelos tengan buen desempeno. Revisa las alertas, las conclusiones y docs/publicacion.md al terminar.


In [1]:
from pathlib import Path
import sys
import json

def _buscar_proyecto(paquete):
    """Encuentra el paquete propio desde el proyecto, tema o portafolio."""
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "src" / paquete).is_dir():
            return base
        for container in (base, base / "Analisis"):
            for pattern in (f"*/src/{paquete}", f"*/*/src/{paquete}"):
                for source in sorted(container.glob(pattern)):
                    parts = source.relative_to(container).parts
                    if "back" in parts or any(p.startswith(".") for p in parts):
                        continue
                    candidate = source.parent.parent
                    if (candidate / "pyproject.toml").is_file():
                        return candidate
    return None

PROJECT_ROOT = _buscar_proyecto("smartfood")
if PROJECT_ROOT is None:
    raise RuntimeError("Abre este notebook desde SmartFood o notebooks.")
TIMEOUT_POR_CELDA = 900
print("Python de la ejecucion:", sys.executable)


Python de la ejecucion: c:\Users\mauro\AppData\Local\Programs\Python\Python314\python.exe


In [2]:
# Ejecutar solo este maestro; no ejecutar otros notebooks en paralelo.
# nbclient inicia y cierra un kernel independiente para cada etapa.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from scripts.execute_notebooks import ejecutar_secuencia
codigo_salida = ejecutar_secuencia(PROJECT_ROOT, timeout=TIMEOUT_POR_CELDA)
if codigo_salida != 0:
    raise RuntimeError("La secuencia se detuvo. Revisa reports/ejecucion/resumen.json y la copia de fallo.")


Ejecutando 01_preparacion_datos.ipynb...
Completado 01_preparacion_datos.ipynb (55.05 s)
Ejecutando 02_analisis_exploratorio.ipynb...
Completado 02_analisis_exploratorio.ipynb (51.32 s)
Ejecutando 03_baseline_perfil_usuario.ipynb...
Completado 03_baseline_perfil_usuario.ipynb (1.31 s)
Ejecutando 04_filtrado_colaborativo_nmf.ipynb...
Completado 04_filtrado_colaborativo_nmf.ipynb (8.37 s)
Ejecutando 05_clasificacion_ranking_svm.ipynb...
Completado 05_clasificacion_ranking_svm.ipynb (3.05 s)
Los cinco notebooks terminaron sin errores de ejecución.


In [3]:
# Esta celda tambien puede ejecutarse para consultar el avance o un fallo.
import pandas as pd
from IPython.display import display
ruta_resumen = PROJECT_ROOT / "reports" / "ejecucion" / "resumen.json"
if not ruta_resumen.exists():
    raise FileNotFoundError("Todavia no existe un resumen de ejecucion.")
resumen = json.loads(ruta_resumen.read_text(encoding="utf-8"))
print("Estado:", resumen["estado"])
display(pd.DataFrame(resumen["notebooks"]))
from smartfood.informe import generar_informe
from IPython.display import Markdown
informe = generar_informe(PROJECT_ROOT)
display(Markdown("[Abrir informe integral](../reports/informe_final/informe_final.md)"))


Estado: completado


,notebook,estado,duracion_segundos
0,01_preparacion_datos.ipynb,completado,55.05
1,02_analisis_exploratorio.ipynb,completado,51.32
2,03_baseline_perfil_usuario.ipynb,completado,1.31
3,04_filtrado_colaborativo_nmf.ipynb,completado,8.37
4,05_clasificacion_ranking_svm.ipynb,completado,3.05


[Abrir informe integral](../reports/informe_final/informe_final.md)